In [3]:
import os
from typing import List, TypedDict, Union
from langchain.chat_models import init_chat_model
from langchain.messages import HumanMessage, AIMessage
from langgraph.graph import StateGraph, START, END 
from dotenv import load_dotenv

In [4]:
load_dotenv()

True

In [7]:
class AgentState(TypedDict):
    messages: List[Union[HumanMessage, AIMessage]]

llm = init_chat_model("google_genai:gemini-2.5-flash-lite")

def process(state: AgentState) -> AgentState:
    """This node will solve the request you input"""
    response =  llm.invoke(state["messages"])
    state["messages"].append(AIMessage(content=response.content))

graph = StateGraph(AgentState)
graph.add_node("process", process)
graph.add_edge(START, "process")
graph.add_edge("process", END)
agent = graph.compile()

conversation_history = []

user_input = input("Enter: ")

while user_input != "exit":
    conversation_history.append(HumanMessage(content=user_input))
    result = agent.invoke({"messages": conversation_history})
    print(result["messages"])
    conversation_history = result["messages"]
    user_input = input("Enter: ")

with open("logging.txt", "w") as file:
    file.write("Your conversation log:\n")

    for message in conversation_history:
        if isinstance(message, HumanMessage):
            file.write(f"You : {message.content}\n")
        if isinstance(message, AIMessage):
            file.write(f"AI : {message.content}\n\n")

    file.write("End of conversation.")

print("Conversation saved to logging.txt")

[HumanMessage(content='Hello, I am sundar', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Sundar! It's nice to meet you. How can I help you today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
[HumanMessage(content='Hello, I am sundar', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Sundar! It's nice to meet you. How can I help you today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[]), HumanMessage(content='what is my name', additional_kwargs={}, response_metadata={}), AIMessage(content='Your name is Sundar.', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
Conversation saved to logging.txt
